In [ ]:
from itertools import product
import sympy as sp
import numpy as np
from scipy.sparse.linalg import expm, expm_multiply
from scipy.sparse import lil_matrix, identity, csr_matrix, dok_matrix
import matplotlib.pyplot as plt

In [ ]:
pip install --upgrade numba


In [ ]:


def basis_cr2(n):
    basis = []
    L = n + 1  # Length of the spin chain plus one for periodic boundary conditions
    r_spins = list(product([1, 0], repeat=L))  # Generate all possible spin configurations
    
    for state in r_spins:
        # Check for the allowed configurations only
        valid = True
        for j in range(L):
            if state[j] == 1:
                # Configuration should be 101
                j1p = (j + 1) % L
                jp = (j ) 
                #j0p = (j -1) if j>0 else n
                if state[j1p] == 1 and state[jp] == 1 :#and state[j0p] == 0:
                    valid = False
                    break

                else:
                    continue
        
        if valid :#or state.count(1) == 0:
            basis.append(list(state))  # Add valid configuration to the basis
    
    return basis


In [ ]:

def h(bases, mu_omega):
    d = len(bases)
    nj = int()
    bases=np.array(bases)
    
    hmat = lil_matrix((d, d), dtype=np.float64)
    for i, state in enumerate(bases):
        L = len(state)
        for j in range(len(state)):
            n_s = state[j]
            nj += ((-1**j)*n_s) + ((1-(-1**j))/2)
            
            j1p = (j + 1) % L
            jp = (j ) 
            j0p = (j -1) if j>0 else L-1
            
            if state[j1p] == 0 and state[jp] == 1 and state[j0p] == 0:
                new_state = state.copy()
                new_state[jp] = 0
                k = bases.any(new_state)
                hmat[i,k] = 1
                
            if state[j1p] == 0 and state[jp] == 0 and state[j0p] == 0:
                new_state = state.copy()
                new_state[jp] = 1
                k = bases.any(new_state)
                
                hmat[i,k] = 1
                
        
            
        
        hmat[i,i] += mu_omega*nj

    return hmat
        

In [ ]:
bases = basis_cr2(15)
bases[0]

In [ ]:
len(bases)

In [ ]:
#bases = basis_cr2(15)
for it, state in enumerate(bases):
    valid = True
    for itl in range(len(state)):
        j_1 = (itl+1) % len(state)
         
        if state[itl] == state[j_1] :
            valid = False
            
        else:
            
            continue
        
            
    if valid :
        print(it)
        
        
        
        

In [ ]:
bases[0]

In [ ]:
def basis_cr2(lent):
    return [np.array([1, 0]), np.array([0, 1])] * lent

bases = basis_cr2(30)
print(bases)

In [ ]:
import numpy as np
from scipy.linalg import expm
from numba import njit
import matplotlib.pyplot as plt

# Dummy function for basis_cr2, replace with the actual function
#def basis_cr2(lent):
 #   return [np.array([1, 0]), np.array([0, 1])] * lent

# Dummy function for h, replace with the actual function
#def h(bases, mu_omega):
 #   return np.eye(len(bases))

# Precomputations
lent = 15
bases = basis_cr2(lent)
dim = len(bases)
rho0 = np.zeros((dim, dim), dtype=np.complex128)
start_state = 29
rho0[start_state, start_state] = 1
mu_omega = 0
t = np.arange(100)
dt = 1
mat_t = h(bases, mu_omega)


# Precompute matrix exponentials
expms = np.array([expm(-1j * mat_t * n * dt) for n in range(len(t))])

# JIT-compiled function
@njit
def compute_ma(dim, expms, rho0, t, dt, bases, start_state):
    ma = []
    M = 0

    for n in range(len(t)):
        ut = expms[n]
        rhot = ut @ rho0
        rho_st = rhot[:, start_state]

        for i in range(len(rho_st)):
            if rho_st[i] != 0:
                state = bases[i]
                L = len(state)
                nj_sum = 0
                for j in range(L):
                    n_s = (-1)**j * state[j]
                    nj_sum += n_s + (1 - (-1)**j) / 2
                M += (rho_st[i].real * nj_sum) * (1 / (L * 100))
        ma.append(M)
    
    return ma

# Call the JIT-compiled function
ma = compute_ma(dim, expms, rho0, t, dt, np.array(bases), start_state)

# Plotting the results
plt.plot(ma)
plt.xlabel('Time')
plt.ylabel('M')
plt.title('Evolution of M over Time')
plt.show()


In [ ]:
bases = basis_cr2(5)
import sympy as sp
mat_t = h(bases, mu_omega)
ma = (mat_t.todense())
#mas = sp.Matrix(ma)
#mas

In [ ]:
bases

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(diagonal_matrix)

In [ ]:
import numpy as np

# Create your matrix (replace with your actual matrix)
a = ma

# Compute eigenvalues and eigenvectors
eigenvalues, eigenvectors = np.linalg.eig(a)

# Diagonalize the matrix
diagonal_matrix = np.linalg.inv(eigenvectors) @ a @ eigenvectors

print("Eigenvalues:", eigenvalues)
print("Eigenvectors:\n", eigenvectors)
print("Diagonal Matrix:\n", diagonal_matrix)


In [ ]:
import numpy as np
from scipy.linalg import expm
from numba import njit
import matplotlib.pyplot as plt

# Dummy function for basis_cr2, replace with the actual function
def basis_cr2(lent):
    return [np.array([1, 0]), np.array([0, 1])] * lent

# Dummy function for h, replace with the actual function
def h(bases, mu_omega):
    return np.eye(len(bases))

# Precomputations
lent = 15
bases = basis_cr2(lent)
dim = len(bases)
mu_omega = 0
t = np.arange(100)
dt = 0.1
mat_t = h(bases, mu_omega)
s_0 = 2

# Precompute matrix exponentials
expms = np.array([expm(-1j * mat_t * n * dt) for n in range(len(t))])

# JIT-compiled function
@njit
def compute_fid(dim, expms, s_0):
    fid = []
    identity_mat = np.eye(dim, dtype=np.complex128)  # Ensure the identity matrix is complex

    for n in range(len(expms)):
        ut1 = expms[n]
        rhot = ut1 @ identity_mat
        a = rhot[:, s_0]
        ft = a[s_0]
        ftc = np.conj(ft)
        fidel = np.abs(ftc * ft)
        fid.append(fidel)

    return fid

# Call the JIT-compiled function
fid = compute_fid(dim, expms, s_0)

# Convert results to real values
t_values = np.real(fid)

# Plotting the results
plt.plot(t_values)
plt.xlabel('Time')
plt.ylabel('Fidelity')
plt.title('Evolution of Fidelity over Time')
plt.show()


In [ ]:
import cProfile
cProfile.run('compute_ma(dim, mat_t, rho0, t, dt, bases, start_state)')


In [ ]:
from scipy.sparse.linalg import expm, expm_multiply
#lent = 10
#bases = basis_cr2(lent)
dim = len(bases)
rho0 = csr_matrix((dim, dim),dtype = np.int8)
start_state = 0
rho0[start_state,start_state] = 1
mu_omega = 0
t = [i for i in range(100)]

ma= []
nj = int()
M = int()
dt = 0.1
mat_t = h(bases,mu_omega)
#p,d = (sp.Matrix(mat_t.todense())).diagonalize()
for n in range(len(t)):
    
    tlt = -1j*mat_t*n
    ut = expm(tlt)

    #rho0 = identity(len(bases))
    #rho0 = (p.conjugate())*rho0*p
    rhot = (ut*rho0)
    #rhot_spm = sp.Matrix(np.real(rhot.todense())) 
    rho_st = rhot[:,start_state]

    for i in range(len(rho_st.todense())):
        
        if rho_st[i] != 0:
            state = bases[i]
            L = len(bases[i])
            for j in range(L):
            
                n_s = (-1**j)*state[j]
                nj = (n_s) + ((1-(-1**j))/2)
                M += ((rho_st[i])*nj)*(1/(L*100))
    ma.append(M)


    
print(M)    
plt.plot(ma)    

In [ ]:
fid = []
s_0 = 10
for n in range(len(t)):
    tlt = -1j*mat_t*n*0.1
    ut1 = expm(tlt)
    #rho0 = identity(len(bases))
    rho0 =  dok_matrix((dim, dim), dtype=np.float32)
    rho0[start_state,start_state] = 1
    rhot = ut1*rho0
    rhot_spm = sp.Matrix((np.real(rhot)).todense())
    a = rhot[:,s_0]
    ft = a[s_0]
    ftc = ft.conjugate()
    fidel = abs(ftc*ft)
    fid.append(fidel)


l =[]
for state in fid:
    c = state.todense()
    b = c[0,0]
    l.append(b)
t = np.real(l)

plt.plot(t,fid)

In [ ]:
ps1 = basis_cr2(15)
print(len(ps1))
print(ps1.index(r))

In [ ]:
from scipy.sparse.linalg import expm, expm_multiply
lent = 10
bases = basis_cr2(lent)
mu_omega = 0
t = [i for i in range(100)]
start_state = 3
ma= []
nj = int()
M = int()
dt = 0.1
mat_t = h(bases,mu_omega)
#p,d = (sp.Matrix(mat_t.todense())).diagonalize()
for n in range(len(t)):
    
    tlt = -1j*mat_t*n
    ut = expm(tlt)
    rho0 = identity(len(bases))
    #rho0 = (p.conjugate())*rho0*p
    rhot = ut*rho0
    rhot_spm = sp.Matrix((np.real(rhot)).todense()) 
    rho_st = rhot_spm[:,start_state]

    for i in range(len(rho_st)):
        state = bases[i]
        if rho_st[i] != 0:
            L = len(bases[i])
            for j in range(L):
            
                n_s = (-1**j)*state[j]
                nj = (n_s) + ((1-(-1**j))/2)
                M += ((-1**j)*(rho_st[i])*nj)*(1/(L*100))
    ma.append(M)


    
print(M)    
plt.plot(ma)    

In [ ]:
fid = []
s_0 = 0
for n in range(len(t)):
    tlt = -1j*mat_t*n*0.1
    ut1 = expm(tlt)
    rho0 = identity(len(bases))
    rhot = ut1*rho0
    rhot_spm = sp.Matrix((np.real(rhot)).todense())
    a = rhot[:,s_0]
    ft = a[s_0]
    ftc = ft.conjugate()
    fidel = abs(ftc*ft)
    fid.append(fidel)


l =[]
for state in fid:
    c = state.todense()
    b = c[0,0]
    l.append(b)
t = np.real(l)

plt.plot(t)